# 🚀 Adaptive Residual Steering (ARS) — Kaggle Execution Runner

**Phase 1: SteerNet-Only (Fixed Layers, No Router, No Gate)**

Pipeline: `Calibration → S1 (SteerNet training) → S5 (deployment calibration) → Benchmark (Baseline vs Fixed-control)`

Goal: SteerNet with fixed layers beats Baseline by >10% on GSM8K.

## 1. Environment & Hardware Check

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas scipy

## 2. Clone Repository (`development` branch)

In [ ]:
import os
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "development"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Per-Layer Norm Calibration (V5)

In [ ]:
import os
os.chdir(REPO_DIR)
print(f"Current directory: {os.getcwd()}")
!python scripts/calibrate_norms.py --n_samples 500 --output norms_calibration.json
!ls -lh norms_calibration.json

## 4. Verify Codebase Integrity & Unit Tests

In [ ]:
import os
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Current directory: {os.getcwd()}")
print(f"Tests directory exists: {os.path.exists('tests/')}")
!pytest tests/ -q --tb=short

## 5. Architecture Configuration

In [ ]:
# v5: rank-64 with causal convolution and per-layer caps
# v4: legacy architecture (default, for NLL training)
STEERNET_ARCH_ARGS = "--steernet_arch v5"
print(f"Architecture: {STEERNET_ARCH_ARGS}")

## 5b. Clear Stale Cache (Phase 1A→1B BUG-6 Fix)

In [ ]:
# ========== PHASE 1A→1B BUG-6 FIX: Clear Stale Cache ==========
# Cache fingerprint was missing κ and architecture, causing stale output reuse.
# This is now fixed in benchmark.py, but we clear the cache to be safe.
import shutil
import os

cache_dir = "./cache/benchmark_outputs"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ Cleared stale cache: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)
else:
    print(f"✅ Cache directory clean: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)

## 6. Hyperparameter Configuration (Phase 1A→1B Updated)

In [ ]:
# ==============================================================================
# ARS EXPERIMENT SCALE & HYPERPARAMETER CONFIGURATION
# Updated with Phase 1A→1B Bug Fixes
# ==============================================================================
# Select execution profile:
#   'balanced' (Recommended): 3000 train rows, 200 test items per shot (~15-18 min)
#   'medium'                : 3000 train rows, 100 test items per shot (~10 min)
#   'full'                  : 7000 train rows, 300 test items per shot (~25 min)
#   'quick'                 : smoke test (~2 min)
PROFILE = 'balanced'
S1_OBJECTIVE = 'nll'  # BUG-8 fix: Always use 'nll', never 'pref' (DPO trap)

# ========== PHASE 1A→1B BUG FIXES ==========
# BUG-2 fix: Single amplitude knob (eliminates dead budget_power parameter)
STEER_KAPPA = 0.025      # Conservative start - real amplitude ~0.025×N_l
K_SCALE_MODE = 'none'    # Disable sqrt scaling (was causing 0.0139×N_l instead of intended amplitude)
STEER_SCALE = 1.0        # Global scale (kept at 1.0, amplitude controlled by KAPPA)

# BUG-5 fix: Fixed-route training (train cooperation, not substitution)
S1_K_CHOICES = [3, 3]     # Train ONLY on deployed route [13,20,22] - 100% of batches

if PROFILE == 'quick':
    S1_EPOCHS     = 2
    S1_LR         = 1e-4
    S1_TRAIN_N    = 50
    S1_VAL_N      = 10
    CALIB_SEL_N   = 5
    BENCH_N_TEST  = 10
    BENCH_SHOTS   = '0'
    BATCH_SIZE    = 4
elif PROFILE == 'balanced':
    S1_EPOCHS     = 4
    S1_LR         = 1e-4
    S1_TRAIN_N    = 3000
    S1_VAL_N      = 100
    CALIB_SEL_N   = 60
    BENCH_N_TEST  = 200
    BENCH_SHOTS   = '0 2 4'
    BATCH_SIZE    = 4
elif PROFILE == 'medium':
    S1_EPOCHS     = 4
    S1_LR         = 1e-4
    S1_TRAIN_N    = 3000
    S1_VAL_N      = 100
    CALIB_SEL_N   = 60
    BENCH_N_TEST  = 100
    BENCH_SHOTS   = '0 2 4'
    BATCH_SIZE    = 4
elif PROFILE == 'full':
    S1_EPOCHS     = 4
    S1_LR         = 1e-4
    S1_TRAIN_N    = 7000
    S1_VAL_N      = 120
    CALIB_SEL_N   = 120
    BENCH_N_TEST  = 300
    BENCH_SHOTS   = '0 2 4'
    BATCH_SIZE    = 4

print('='*80)
print('✅ PHASE 1A→1B CONFIGURATION LOADED')
print('='*80)
print(f'Profile: {PROFILE}, Objective: {S1_OBJECTIVE}')
print(f'\n🔧 Bug Fixes Applied:')
print(f'  • BUG-2: Single amplitude KAPPA={STEER_KAPPA}, K_SCALE_MODE={K_SCALE_MODE}')
print(f'  • BUG-5: Fixed-route training S1_K_CHOICES={S1_K_CHOICES}')
print(f'  • BUG-6: Cache cleared + fingerprint fixed in codebase')
print(f'  • BUG-1,3,4,7,8,9: Fixed in source code (committed)')
print(f'\n📊 Training Config:')
print(f'  • S1: {S1_EPOCHS} epochs, {S1_TRAIN_N} samples, lr={S1_LR}')
print(f'  • Batch size: {BATCH_SIZE}')
print(f'  • Calibration: {CALIB_SEL_N} samples')
print(f'  • Benchmark: {BENCH_N_TEST} test items, shots=[{BENCH_SHOTS}]')
print(f'\n🎯 Expected: ≥+10pp gain across all shots (0/2/4-shot)')
print('='*80)


## 7. Fixed Layer Configuration (Phase 1A→1B Updated)

In [ ]:
# ==============================================================================
# FIXED LAYER CONFIGURATION (Phase 1A→1B Updated)
# ==============================================================================
# Fixed layers: L13, L15, L18, L20, L22 (catalog candidate layers)
# S_FIXED: Proven 3-layer synergy (Early L13 + Mid-Late L20 + Late Anchor L22)
# Benchmark runs Baseline vs Fixed-control (--arms baseline,fixed)
# ==============================================================================

FIXED_LAYERS = [13, 15, 18, 20, 22]
CATALOG_MAX_K = 3  # Optimal K (at most 3 layers to avoid noise accumulation)
S_FIXED = '13,20,22'  # Proven 3-layer synergy: L13 (early), L20 (mid-late reasoning), L22 (late anchor)

# --- Steering Physics (Phase 1A→1B Fixes Applied) ---
# [1] Question Prefill Steering: ACTIVE (conditions KV-cache in math space for +10 to +25 pp in 0-shot)
# [2] Few-Shot Prompt Shielding: ACTIVE (demonstration tokens 100% pristine) [BUG-3 FIX]
# [3] Shot-Aware Scale: REMOVED (eliminated train/eval mismatch) [BUG-6 FIX]
# [4] S5 Envelope Calibration: Calibrated under S_FIXED with correct κ [BUG-6 FIX]

print('='*80)
print('✅ FIXED LAYER CONFIGURATION')
print('='*80)
print(f'Fixed candidate layers: {FIXED_LAYERS}')
print(f'Catalog max_k: {CATALOG_MAX_K}')
print(f'Enforced 3-layer subset (S_FIXED): {S_FIXED}')
print(f'\nPhase 1 Constraints:')
print(f'  • Router: DISABLED (Phase 2 feature)')
print(f'  • Gate: DISABLED (Phase 3 feature)')
print(f'\n🔬 Steering Physics (Phase 1A→1B):')
print(f'  • ✅ Question Prefill Steering: ACTIVE')
print(f'  • ✅ Few-Shot Demo Shielding: ACTIVE (BUG-3 fixed)')
print(f'  • ✅ Shot-Aware Scale: REMOVED (BUG-6 fixed, no train/eval mismatch)')
print(f'  • ✅ S5 Envelope: Calibrated under S_FIXED with κ={STEER_KAPPA}')
print(f'  • ✅ Sink Token Protection: ACTIVE (BUG-3 fixed)')
print(f'  • ✅ Gold Annotations: STRIPPED from training (BUG-4 fixed)')
print('='*80)


## 8. Run Full Pipeline: S1 → S5 → Benchmark

In [ ]:
# Run complete pipeline in one command to maintain state between stages
# S1 (Training) → S5 (Calibration) → bench (Benchmark)
#
# Phase 1A→1B Bug Fixes Applied:
# - BUG-2: Single amplitude knob (STEER_KAPPA + K_SCALE_MODE='none')
# - BUG-5: Fixed-route training (S1_K_CHOICES=[3,3] trains only deployed route)
# - BUG-6: Cache cleared, fingerprint fixed
# - BUG-1,3,4,7,8,9: Fixed in source code

!python scripts/run_ars_pipeline.py --stage S1,S5,bench {STEERNET_ARCH_ARGS} \
    --objective {S1_OBJECTIVE} \
    --s1_epochs {S1_EPOCHS} \
    --s1_lr {S1_LR} \
    --s1_train_n {S1_TRAIN_N} \
    --s1_val_n {S1_VAL_N} \
    --calib_sel_n {CALIB_SEL_N} \
    --n_test {BENCH_N_TEST} \
    --shots {BENCH_SHOTS} \
    --batch_size {BATCH_SIZE} \
    --steer_scale {STEER_SCALE} \
    --s_fixed {S_FIXED} \
    --contract deployment_contract.json \
    --record experiment_record.json \
    --report \
    --arms baseline,fixed \
    --no_revert \
    --set catalog_max_k={CATALOG_MAX_K} \
    --set steer_kappa={STEER_KAPPA} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set s1_k_choices={','.join(map(str, S1_K_CHOICES))}


## 9. Inspect Benchmark Report

In [ ]:
import os
report_path = "report/REPORT.md"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Report not found. Check benchmark logs above.")

## 10. Save Artifacts

In [ ]:
import shutil
os.makedirs("/kaggle/working/ars_artifacts", exist_ok=True)
for fname in ["deployment_contract.json", "experiment_record.json", "checkpoints/pipeline_state.json", "norms_calibration.json"]:
    if os.path.exists(fname):
        shutil.copy(fname, "/kaggle/working/ars_artifacts/")
if os.path.exists("report"):
    shutil.copytree("report", "/kaggle/working/ars_artifacts/report", dirs_exist_ok=True)
print("Artifacts saved to /kaggle/working/ars_artifacts:")
!ls -lh /kaggle/working/ars_artifacts